In [2]:
# Import thư viện
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import f_regression
from sklearn.metrics.pairwise import rbf_kernel
import warnings

# file module
from data_pipeline import AirQualityPipeline
from advanced_methods import BayesianLinearRegression

warnings.filterwarnings('ignore')
print("Import thư viện thành công")

Import thư viện thành công


In [3]:
# Tải dữ liệu và tạo Lag Features
print("Đang xử lý dữ liệu và tạo đặc trưng chuỗi thời gian (Lag Features)...")
df_god = pd.read_csv('data/city_day.csv')
df_god = df_god.dropna(subset=['AQI'])
df_god['Date'] = pd.to_datetime(df_god['Date'])
df_god = df_god.sort_values(by=['City', 'Date'])

# Tạo biến trễ thời gian
df_god['AQI_Yesterday'] = df_god.groupby('City')['AQI'].shift(1)
df_god['PM2.5_Yesterday'] = df_god.groupby('City')['PM2.5'].shift(1)
df_god = df_god.dropna(subset=['AQI_Yesterday', 'PM2.5_Yesterday'])

# Lấy mẫu 5000 dòng
df_god = df_god.sample(n=5000, random_state=42).reset_index(drop=True)

# Tách X, y và Logarit
X_god = df_god.drop(columns=['AQI', 'AQI_Bucket'])
y_god = np.log1p(df_god['AQI']) 

# Chia Train/Test
X_tr_raw, X_te_raw, y_tr_god, y_te_god = train_test_split(X_god, y_god, test_size=0.2, random_state=42)
print(f"Kích thước tập Train: {X_tr_raw.shape}, tập Test: {X_te_raw.shape}")

Đang xử lý dữ liệu và tạo đặc trưng chuỗi thời gian (Lag Features)...
Kích thước tập Train: (4000, 16), tập Test: (1000, 16)


In [4]:
# Tiền xử lí 
print("Đang chạy Data Pipeline (Xử lý Missing, Log, Scaling, Encoding)...")
num_cols_god = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI_Yesterday', 'PM2.5_Yesterday']
skewed_cols_god = ['PM2.5', 'PM10', 'NO', 'NOx', 'NH3', 'CO', 'Benzene', 'Toluene', 'Xylene', 'AQI_Yesterday', 'PM2.5_Yesterday']

pipeline_god = AirQualityPipeline(num_cols=num_cols_god, skewed_cols=skewed_cols_god)
X_tr_df = pipeline_god.fit_transform(X_tr_raw)
X_te_df = pipeline_god.transform(X_te_raw)

# Chuyển đổi sang numpy array để tính toán ma trận nhanh hơn
X_tr_proc = X_tr_df.values
X_te_proc = X_te_df.values
y_tr_god = y_tr_god.values.flatten()
y_te_god = y_te_god.values.flatten()
feature_names = X_tr_df.columns.tolist()

print(f"Hoàn tất! Số lượng đặc trưng sau xử lý: {X_tr_proc.shape[1]}")

Đang chạy Data Pipeline (Xử lý Missing, Log, Scaling, Encoding)...
Hoàn tất! Số lượng đặc trưng sau xử lý: 30


In [5]:
# ==========================================
# CELL 4: HÀM ĐÁNH GIÁ & HUẤN LUYỆN 4 MÔ HÌNH ĐẦU
# ==========================================
# Hàm đánh giá nghịch đảo Logarit về thang đo AQI thực
def calculate_aqi_metrics(y_true_log, y_pred_log):
    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)
    y_pred_real = np.clip(y_pred_real, 0, None)
    
    mae = np.mean(np.abs(y_true_real - y_pred_real))
    rmse = np.sqrt(np.mean((y_true_real - y_pred_real)**2))
    tss = np.sum((y_true_real - np.mean(y_true_real))**2)
    rss = np.sum((y_true_real - y_pred_real)**2)
    r2 = 1 - (rss / tss)
    return mae, rmse, r2

print("Đang huấn luyện nhóm Tuyến tính...")

# 1. OLS Cơ bản
X_tr_mat = np.c_[np.ones((X_tr_proc.shape[0], 1)), X_tr_proc]
X_te_mat = np.c_[np.ones((X_te_proc.shape[0], 1)), X_te_proc]
beta_ols = np.linalg.pinv(X_tr_mat.T @ X_tr_mat) @ X_tr_mat.T @ y_tr_god
y_tr_pred_ols = X_tr_mat @ beta_ols
y_te_pred_ols = X_te_mat @ beta_ols

# 2. OLS Chọn biến (Lọc theo p-value < 0.05)
f_stats, p_values = f_regression(X_tr_proc, y_tr_god)
selected_idx = np.where(p_values < 0.05)[0]
X_tr_sel = np.c_[np.ones((X_tr_proc.shape[0], 1)), X_tr_proc[:, selected_idx]]
X_te_sel = np.c_[np.ones((X_te_proc.shape[0], 1)), X_te_proc[:, selected_idx]]
beta_ols_sel = np.linalg.pinv(X_tr_sel.T @ X_tr_sel) @ X_tr_sel.T @ y_tr_god
y_tr_pred_ols_sel = X_tr_sel @ beta_ols_sel
y_te_pred_ols_sel = X_te_sel @ beta_ols_sel

# 3. Ridge với Cross-Validation k=5
ridge_cv = RidgeCV(alphas=[0.1, 1.0, 5.0, 10.0, 20.0], cv=5)
ridge_cv.fit(X_tr_proc, y_tr_god)
best_lambda = ridge_cv.alpha_
y_tr_pred_ridge = ridge_cv.predict(X_tr_proc)
y_te_pred_ridge = ridge_cv.predict(X_te_proc)

# 4. Bayesian Tuyến tính
bayes_model = BayesianLinearRegression(alpha=1.0, beta=10.0).fit(X_tr_df, y_tr_god)
y_tr_pred_bayes, _, _ = bayes_model.get_credible_interval(X_tr_df)
y_te_pred_bayes, _, _ = bayes_model.get_credible_interval(X_te_df)

print(f"Xong! Ridge đã chọn lambda tốt nhất là: {best_lambda}")

Đang huấn luyện nhóm Tuyến tính...
Xong! Ridge đã chọn lambda tốt nhất là: 10.0


In [6]:
# Hàm đánh giá và Huấn luyện 4 mô hình tuyến tính (OLS, OLS chọn biến, Ridge CV, Bayesian)

# Hàm đánh giá nghịch đảo Logarit về thang đo AQI thực
def calculate_aqi_metrics(y_true_log, y_pred_log):
    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)
    y_pred_real = np.clip(y_pred_real, 0, None)
    
    mae = np.mean(np.abs(y_true_real - y_pred_real))
    rmse = np.sqrt(np.mean((y_true_real - y_pred_real)**2))
    tss = np.sum((y_true_real - np.mean(y_true_real))**2)
    rss = np.sum((y_true_real - y_pred_real)**2)
    r2 = 1 - (rss / tss)
    return mae, rmse, r2

print("Đang huấn luyện nhóm Tuyến tính...")

# 1. OLS Cơ bản
X_tr_mat = np.c_[np.ones((X_tr_proc.shape[0], 1)), X_tr_proc]
X_te_mat = np.c_[np.ones((X_te_proc.shape[0], 1)), X_te_proc]
beta_ols = np.linalg.pinv(X_tr_mat.T @ X_tr_mat) @ X_tr_mat.T @ y_tr_god
y_tr_pred_ols = X_tr_mat @ beta_ols
y_te_pred_ols = X_te_mat @ beta_ols

# 2. OLS Chọn biến (Lọc theo p-value < 0.05)
f_stats, p_values = f_regression(X_tr_proc, y_tr_god)
selected_idx = np.where(p_values < 0.05)[0]
X_tr_sel = np.c_[np.ones((X_tr_proc.shape[0], 1)), X_tr_proc[:, selected_idx]]
X_te_sel = np.c_[np.ones((X_te_proc.shape[0], 1)), X_te_proc[:, selected_idx]]
beta_ols_sel = np.linalg.pinv(X_tr_sel.T @ X_tr_sel) @ X_tr_sel.T @ y_tr_god
y_tr_pred_ols_sel = X_tr_sel @ beta_ols_sel
y_te_pred_ols_sel = X_te_sel @ beta_ols_sel

# 3. Ridge với Cross-Validation k=5
ridge_cv = RidgeCV(alphas=[0.1, 1.0, 5.0, 10.0, 20.0], cv=5)
ridge_cv.fit(X_tr_proc, y_tr_god)
best_lambda = ridge_cv.alpha_
y_tr_pred_ridge = ridge_cv.predict(X_tr_proc)
y_te_pred_ridge = ridge_cv.predict(X_te_proc)

# 4. Bayesian Tuyến tính
bayes_model = BayesianLinearRegression(alpha=1.0, beta=10.0).fit(X_tr_df, y_tr_god)
y_tr_pred_bayes, _, _ = bayes_model.get_credible_interval(X_tr_df)
y_te_pred_bayes, _, _ = bayes_model.get_credible_interval(X_te_df)

print(f"Ridge đã chọn lambda tốt nhất là: {best_lambda}")

Đang huấn luyện nhóm Tuyến tính...
Ridge đã chọn lambda tốt nhất là: 10.0


In [7]:
# Kernel RBF grid search

print("Đang quét tham số tối ưu cho Kernel RBF...")
best_te_r2_kernel = -float('inf')
best_y_tr_kernel, best_y_te_kernel = None, None
best_g, best_l = 0.0, 0.0

for g in [0.001, 0.005, 0.01]:
    for l in [0.1, 1.0, 5.0]:
        K_tr_temp = rbf_kernel(X_tr_proc, X_tr_proc, gamma=g)
        A_temp = K_tr_temp + l * np.eye(X_tr_proc.shape[0])
        alpha_temp = np.linalg.solve(A_temp, y_tr_god)
        
        K_te_temp = rbf_kernel(X_te_proc, X_tr_proc, gamma=g)
        y_te_pred_temp = K_te_temp @ alpha_temp
        
        _, _, te_r2 = calculate_aqi_metrics(y_te_god, y_te_pred_temp)
        if te_r2 > best_te_r2_kernel:
            best_te_r2_kernel = te_r2
            best_y_tr_kernel = K_tr_temp @ alpha_temp
            best_y_te_kernel = y_te_pred_temp
            best_g, best_l = g, l

print(f"Cấu hình an toàn cho Kernel RBF: Gamma={best_g}, Lambda={best_l}")

Đang quét tham số tối ưu cho Kernel RBF...
Cấu hình an toàn cho Kernel RBF: Gamma=0.005, Lambda=0.1


In [8]:
# Tổng hợp bảng so sánh MAE, RMSE, R2
models_name = [
    '1. OLS Cơ bản', 
    '2. OLS Chọn biến (p-value < 0.05)', 
    f'3. Ridge CV (lambda={best_lambda})', 
    '4. Bayesian Tuyến tính', 
    f'5. Kernel RBF (g={best_g}, l={best_l})'
]

y_train_preds = [y_tr_pred_ols, y_tr_pred_ols_sel, y_tr_pred_ridge, y_tr_pred_bayes, best_y_tr_kernel]
y_test_preds = [y_te_pred_ols, y_te_pred_ols_sel, y_te_pred_ridge, y_te_pred_bayes, best_y_te_kernel]

train_results, test_results = [], []
for tr_p, te_p in zip(y_train_preds, y_test_preds):
    train_results.append(calculate_aqi_metrics(y_tr_god, tr_p))
    test_results.append(calculate_aqi_metrics(y_te_god, te_p))

columns = pd.MultiIndex.from_product([['Tập Huấn Luyện', 'Tập Kiểm Thử'], ['MAE', 'RMSE', 'R-squared']])
final_report = pd.DataFrame(np.hstack((train_results, test_results)), index=models_name, columns=columns)

print("\n" + "="*80)
print("BẢNG TỔNG HỢP KIỂM ĐỊNH HIỆU NĂNG CHO CÁC MÔ HÌNH")
print("="*80)
display(final_report)


BẢNG TỔNG HỢP KIỂM ĐỊNH HIỆU NĂNG CHO CÁC MÔ HÌNH


Tập Huấn Luyện                       \
                                             MAE       RMSE R-squared   
1. OLS Cơ bản                          23.888258  48.070031  0.878252   
2. OLS Chọn biến (p-value < 0.05)      23.899158  47.895324  0.879135   
3. Ridge CV (lambda=10.0)              23.926603  48.316118  0.877002   
4. Bayesian Tuyến tính                 23.891107  48.080142  0.878201   
5. Kernel RBF (g=0.005, l=0.1)         18.778162  34.459285  0.937436   

                                  Tập Kiểm Thử                       
                                           MAE       RMSE R-squared  
1. OLS Cơ bản                        26.405511  58.034898  0.832531  
2. OLS Chọn biến (p-value < 0.05)    26.374403  57.857575  0.833553  
3. Ridge CV (lambda=10.0)            26.440864  58.152796  0.831850  
4. Bayesian Tuyến tính               26.408401  58.045195  0.832472  
5. Kernel RBF (g=0.005, l=0.1)       21.179390  50.026272  0.875562